# S05 — E3 results (extended, consumption-based CI)

Trains and evaluates the **E3** model: the CarbonCast-*extended* pipeline that
retargets to consumption-based CI and adds cross-border flow ANNs plus partner-CI
channels. Imports the `carboncast_extended` orchestrator; never reimplements model
logic. Loads materialized `data/processed/{zone}.parquet` frames, so it is portable
to Colab.

E3 is the thesis model (Contributions 1-2). Its production-based counterpart E2
lives in S04; note the two optimize **different targets**, so MAPE is not directly
comparable across the two notebooks (see the comparability note below).

## 1. Setup

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_ROOT = "/content/drive/MyDrive/carbon-intensity-forecast/data"
else:
    sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
    DATA_ROOT = os.path.abspath(os.path.join("..", "data"))

os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import time
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from carbon_forecast.data import storage
from carbon_forecast.plotting import config as P
from carbon_forecast.models.carboncast_extended import (
    train_e3, evaluate_ci, predict_with_truth, E3Config,
)
from carbon_forecast.models.tier1_source import Tier1Config
from carbon_forecast.models.tier1_flow import Tier1FlowConfig
from carbon_forecast.models.tier2_cnnlstm import Tier2Config

P.apply_defaults()
print("Colab:", IN_COLAB, "| DATA_ROOT:", DATA_ROOT)

## 2. Data and split

Locked split is Train 2021-2025, Validation 2026 Jan-Apr, Test A 2026 May, Test B
2026 Jun 8-21. 2026 history is not yet extracted, so this uses the **proxy split**
(Train 2021-23, Val 2024, Test 2025), matching S04 so E2 and E3 are run on the same
folds. Switch the date strings once 2026 is pulled.

In [ ]:
ZONES = ["BE", "FI", "SG", "US-MIDA-PJM", "US-NY-NYIS"]

def load_zone(zone):
    return storage.read_parquet(storage.processed_path(zone, DATA_ROOT))

SPLIT = dict(train=slice("2021", "2023"), val=slice("2024", "2024"), test=slice("2025", "2025"))

be = load_zone("BE")
train, val, test = be.loc[SPLIT["train"]], be.loc[SPLIT["val"]], be.loc[SPLIT["test"]]
for name, part in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:5s} {part.index.min().date()} -> {part.index.max().date()}  ({len(part):,} h)")

## 3. Train E3 (BE)

Full settings, matching S04: 100-epoch cap with early stopping (stride 1) for the
final source and flow models and the Tier 2 CNN-LSTM; out-of-fold fold models use a
fixed 60-epoch budget. Source and flow Tier 1 models both run out-of-fold for honest
Tier 2 training; partner CI is held to actuals in training (strategy A).

In [ ]:
cfg = E3Config(
    tier1=Tier1Config(epochs=100, stride=1, patience=10),
    tier1_fold=Tier1Config(epochs=60, stride=1),
    flow=Tier1FlowConfig(epochs=100, stride=1, patience=10),
    flow_fold=Tier1FlowConfig(epochs=60, stride=1),
    tier2=Tier2Config(epochs=100, stride=1, patience=10),
)

t0 = time.time()
art = train_e3(train, val, "BE", cfg, verbose=1)
print(f"\ntrained BE E3 in {(time.time() - t0) / 60:.1f} min")
print(f"dynamic channels: {len(art.dynamic_cols)} = {len(art.source_cols)} source "
      f"+ {len(art.flow_cols)} flow + {len(art.partner_ci_cols)} partner-CI")

## 4. Evaluate on the test fold

In [ ]:
metrics = evaluate_ci(art, test)
print("E3 — BE test (consumption-based CI):")
print(f"  MAPE : {metrics['mape_pct']:.2f} %   (primary)")
print(f"  MAE  : {metrics['mae']:.2f} gCO2eq/kWh")
print(f"  RMSE : {metrics['rmse']:.2f} gCO2eq/kWh")
print(f"  n    : {metrics['n_samples']:,} forecast origins")

## 5. Per-horizon degradation curve (full 1-96 h)

Error as a function of forecast horizon, every hour from 1 to 96. Shown for both
MAPE (primary, scale-relative) and MAE (absolute). If a feature appears in MAPE but
not MAE it is a denominator effect (consumption-based CI dipping low at that
horizon), not a genuine rise in error. The 1-24 h band is where the EM head-to-head
will later overlay (academic-key forecast horizon).

In [ ]:
preds, y_true, origins = predict_with_truth(art, test)
ae = np.abs(preds - y_true)
mape_h = np.nanmean(ae / np.clip(np.abs(y_true), 1e-6, None), axis=0) * 100
mae_h = np.nanmean(ae, axis=0)
hours = np.arange(1, preds.shape[1] + 1)

# Quick text diagnostic of the tail.
print("MAPE  h1/h24/h48/h72/h96:", [round(float(mape_h[h-1]), 1) for h in [1,24,48,72,96]])
print("MAE   h1/h24/h48/h72/h96:", [round(float(mae_h[h-1]), 1) for h in [1,24,48,72,96]])
print(f"argmax MAPE at h{int(mape_h.argmax())+1} ({mape_h.max():.1f}%); "
      f"argmax MAE at h{int(mae_h.argmax())+1} ({mae_h.max():.1f})")

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12,
                    subplot_titles=("MAPE (%)", "MAE (gCO2eq/kWh)"))
fig.add_trace(go.Scatter(x=hours, y=mape_h, mode="lines",
                         line=dict(color=P.MODEL_PALETTE["open"], width=P.LINE_WIDTH),
                         name="E3 MAPE"), row=1, col=1)
fig.add_trace(go.Scatter(x=hours, y=mae_h, mode="lines",
                         line=dict(color=P.CI_COLOR, width=P.LINE_WIDTH),
                         name="E3 MAE"), row=2, col=1)
for r in (1, 2):
    for h in (24, 48, 72):
        fig.add_vline(x=h, line=dict(color="rgba(0,0,0,0.15)", width=1), row=r, col=1)
fig.update_xaxes(title="forecast horizon (hours ahead)", row=2, col=1)
P.style_fig(fig, "E3 carbon-intensity error by forecast horizon",
            subtitle="BE, consumption-based CI, test 2025 - full 1-96 h", height=650)
fig.show()

## 6. E2 vs E3 comparability

E2 (S04) targets production-based CI; E3 targets consumption-based CI. They are
errors on **different series**, so a direct MAPE comparison is misleading. The
meaningful read is (a) E3's absolute error (MAE/RMSE) on the operationally relevant
consumption-based target, and (b) the shape of the per-horizon curve above. A
like-for-like comparison is reserved for the EM head-to-head (Contribution 4), where
both E3 and EM's operational forecast are scored against the same consumption-based
ground truth on the 1-24 h band.

## 7. All five zones

Trains E3 across every zone and collects headline metrics. This is the heavy run
(source + flow Tier 1, out-of-fold, x5 zones) - a good candidate for the Colab Pro
GPU. Uncomment to run.

In [ ]:
# results = {}
# for zone in ZONES:
#     df = load_zone(zone)
#     ztr, zva, zte = df.loc[SPLIT['train']], df.loc[SPLIT['val']], df.loc[SPLIT['test']]
#     t0 = time.time()
#     zart = train_e3(ztr, zva, zone, cfg, verbose=0)
#     results[zone] = {**evaluate_ci(zart, zte), 'minutes': (time.time()-t0)/60}
#     print(f"{zone:14s} MAPE={results[zone]['mape_pct']:.2f}%  ({results[zone]['minutes']:.1f} min)")
# pd.DataFrame(results).T